# Text Classification

> Single-label and multi-label text classification with the v2 `TextClassificationLearner`.

This notebook follows a progressive path: **Quick Start** first, then deeper usage patterns, then a compact API reference.

## Notebook Metadata

`nbdev` export target for this notebook.

In [ ]:
#| default_exp text.classification


## Setup

Import the learner and common dependencies used in all examples.

In [ ]:
import warnings

warnings.filterwarnings('ignore')

import pandas as pd

from blurr.text.classification import TextClassificationLearner


## Quick Start

Minimal single-label sentiment classification from a Hugging Face dataset.

Use this pattern when you want the fastest path from raw data to predictions.

In [ ]:
learn = TextClassificationLearner.from_hf(
    'imdb',
    model_name='prajjwal1/bert-tiny',
    text_col='text',
    label_col='label',
    max_samples=200,
    max_length=128,
    bs=16,
)

learn.fit_one_cycle(1, 2e-5)
quick_pred = learn.blurr_predict('This movie was unexpectedly good!')[0]
quick_pred


**Expected output**

A dictionary with keys like `label`, `score`, `probs`, and `class_labels`.
For example, a positive review should typically produce a high-confidence `positive` label.

## Single-Label Classification (Detailed)

This section expands on Quick Start with batch prediction and model behavior inspection.

In [ ]:
batch_preds = learn.blurr_predict([
    'Absolutely loved the performances and soundtrack.',
    'The pacing was slow and the ending was disappointing.',
])

for pred in batch_preds:
    print(f"label={pred['label']:<8} score={pred['score']:.4f}")


### From a DataFrame

Use `from_df` when your training data already lives in pandas (CSV/SQL pipelines, annotation tools, etc.).

In [ ]:
df = pd.DataFrame({
    'text': [
        'Great product and fast shipping.',
        'Poor build quality and broke quickly.',
        'Works exactly as expected.',
        'Customer support was unhelpful.',
        'Excellent value for the price.',
        'Not worth the money.',
    ],
    'sentiment': ['positive', 'negative', 'positive', 'negative', 'positive', 'negative'],
})

learn_df = TextClassificationLearner.from_df(
    df,
    model_name='prajjwal1/bert-tiny',
    text_col='text',
    label_col='sentiment',
    bs=4,
    max_length=96,
)

learn_df.fit_one_cycle(1, 2e-5)


## Multi-Label Classification

Pass a list of binary label columns and set `is_multilabel=True`.

This is useful for toxicity, topic tagging, intent routing, or any scenario where each input can have **multiple** labels.

In [ ]:
ml_df = pd.DataFrame({
    'text': [
        'I appreciate your help!',
        'You are clueless and rude.',
        'This is hateful language.',
        'Great teamwork today.',
    ],
    'toxic':  [0, 1, 1, 0],
    'insult': [0, 1, 0, 0],
    'hate':   [0, 0, 1, 0],
})

learn_multi = TextClassificationLearner.from_df(
    ml_df,
    model_name='prajjwal1/bert-tiny',
    text_col='text',
    label_col=['toxic', 'insult', 'hate'],
    is_multilabel=True,
    bs=4,
    max_length=96,
)

learn_multi.fit_one_cycle(1, 2e-5)


## Advanced Usage

`from_data` auto-dispatches to DataFrame/HF/file-path constructors, which is handy in reusable training scripts.

In [ ]:
learn_auto = TextClassificationLearner.from_data(
    df,
    model_name='prajjwal1/bert-tiny',
    text_col='text',
    label_col='sentiment',
    bs=4,
)

auto_pred = learn_auto.blurr_predict('Mediocre quality but acceptable for the price.')
auto_pred[0]


## API Reference

### Main class
- `TextClassificationLearner`

### Construction methods
- `from_hf(dataset, model_name, ...)`
- `from_df(df, model_name, ...)`
- `from_data(data, model_name, ...)`

### Inference
- `blurr_predict(texts)` → list of prediction dictionaries

### Training
Inherits fastai `Learner` methods (`fit_one_cycle`, `fine_tune`, `lr_find`, `freeze`, `unfreeze`, ...).
